# E3 · Piloto v2 · CNN-LSTM con atención · Resultados en GitHub

Se conserva la misma **CNN causal → LSTM → atención temporal** de M00. La variante v2 aprende una escala de magnitud de innovación y la transforma en una predicción de σ₇ que respeta el piso ya observado. Se compara con **cinco controles**; tres redes nuevas comparten arquitectura e inicialización.

1. En **Entorno de ejecución → Cambiar tipo de entorno**, selecciona GPU; se recomienda la **A100**. Se verifica el hardware realmente asignado.
2. Ejecuta las celdas en orden. La instalación aislada de Python y CUDA puede tardar varios minutos y se reutiliza en esta sesión.
3. Habilita el secreto de Colab **`GITHUB_TOKEN`**, con permiso **Contents: Read and write** para `jassan-bit/DeepLearning_202630`. Si falta, se solicita en un campo oculto. Permanece en memoria y no entra al proceso científico.
4. Los archivos se conservan en **`e3-desarrollo-resultados-v2/runs/innovacion_v2`**. Una reconexión restaura esa misma corrida, sus pesos y el gasto acumulado.
5. El tope adicional conjunto es **600 segundos**, incluyendo ajustes iniciales, entrenamiento, validación y checkpoints. El cargo previo de v1 (**369.48 s**) se conserva dentro del límite global de **1200 s**. Instalación y descargas pueden añadir tiempo de pared.

La validación interna ya fue examinada en desarrollo. Este piloto **no es confirmatorio** y no certifica novedad, calibración o superioridad frente al Top 10. Si alguna red no termina su receta por presupuesto, el resultado se declara incompleto.


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib
import importlib.util
import json
import os
import re
import subprocess
import sys
import urllib.request

REPOSITORY = "jassan-bit/DeepLearning_202630"
SOURCE_COMMIT = "034bbe40e47d4e5f58674b302a079ab7c090f0b7"
RESULTS_BRANCH = "e3-desarrollo-resultados-v2"
RESULTS_URL = f"https://github.com/{REPOSITORY}/tree/{RESULTS_BRANCH}/runs/innovacion_v2"
DEVICE = "cuda"
LOCAL_WORKSPACE = Path("/content/entregable3_innovacion_v2")
SOURCE_DIR = LOCAL_WORKSPACE / ("source-" + SOURCE_COMMIT)
RUN_DIR = LOCAL_WORKSPACE / "github-run-innovacion-v2"
print("Código fijado:", SOURCE_COMMIT)
print("Resultados:", RESULTS_URL)


## Descargar y verificar el código fijado

Cada archivo se comprueba con SHA-256 antes de importarlo. Los datos permanecen en el disco temporal de Colab y sólo los archivos admitidos de la corrida se publican en GitHub.


In [ ]:
SOURCE_FILE_SHA256 = {
    "src/__init__.py": "c7b20782a739f0118bbfbab216b4a3fb323edef4cb5c5eaf7f5cf482aaa18d74",
    "src/original_model.py": "dce48ff772cd284c63a672c599febc66dfc6eb14f286f523703fe6b01d086c2f",
    "src/e3_development.py": "33dd121f0dd36ef1933c5f4338045fa5a17e85ad6986dfd2c1a43c2662158ebe",
    "src/e3_innovation_model.py": "7f362d38ca83ed302dc9c822b4e4e19a0d446e4c01477f42b0e9a8da823ae7b1",
    "src/e3_innovation.py": "3bf0a6ded102e60fb4f3a46d92ca64b41fad393c6d9da56986c889b54789cd25",
    "scripts/e3_run_development.py": "8f100e6aed1ade7336e7c104cefc475b61ee5cd8db20e04a253f099d0190ce7c",
    "scripts/e3_run_innovation.py": "c08b3c3200d7c09ee88e6695ad0755029328fb6c19da4967b6c8651388a8d32a",
    "scripts/e3_github_storage.py": "1de75f622c35e0a9375e96197a620663411243c87c93b1bc2e23d45032f0d632",
    "scripts/e3_innovation_github.py": "ee136af78d7d8f2a851a1ca5708a33e5e4ee2d68f2254002e5d9e4a7190d95ae",
    "scripts/e3_verify_innovation_design.py": "50a68774e265f24a9494e5509af03f4ddfb9a1535ce46a9e4ecaa52dd3762aee",
    "notebooks/preparar_e3_desarrollo_runtime.py": "9b4ef359b2407f197d9841e345f87ea769c72a7aec871b07c9563d9f852d22d0",
    "notebooks/preparar_e3_innovacion_runtime.py": "de3025c5f040cab4f69e1640494423a13a5640eb68999ec343897dfb507cd5e2",
    "requirements-e3-innovation-gpu.lock.txt": "85f80fa265764d6aa213ff4b375f946eb187acb58cb915311f3a5268122224d6",
    "experiments/e3_innovacion_v2_20261116/plan.json": "5dcf1f266b5340328af08ed275bc5351281f52c4a9338f1230c5e09cf90a5ad9",
    "experiments/e3_ficha_cnn_lstm_atencion_v2_20261116/pilot_plan.json": "ef0bb7c763c336e498f32d7bd79a1d757faf37a8c0de5378a3386783ef4ca67a",
    "experiments/e3_ficha_cnn_lstm_atencion_v2_20261116/verification/quadrature_k128.json": "c3ce43d3cb81767ee3fc0daa0ccfe8432a3326643e28e51273a49cc1fe01c2fe",
    "experiments/e3_ficha_cnn_lstm_atencion_v2_20261116/verification/quadrature_k128.json.sha256": "624b0c1dde0cebf59390a921e6e4830ea51dff81658293b329e152f74a7b86ae",
    "experiments/e3_ficha_cnn_lstm_atencion_v2_20261116/verification/summary.json": "e9f9753d8f9e13785dc6de30a4a176f171a8f76d7bf49ce332fdc58a8e46dee8"
}

def download_pinned(relative_path, commit, destination, expected_sha256):
    relative = PurePosixPath(relative_path)
    if (relative.is_absolute() or ".." in relative.parts or "\\" in relative_path
            or str(relative) != relative_path or not re.fullmatch(r"[0-9a-f]{40}", commit)
            or not re.fullmatch(r"[0-9a-f]{64}", expected_sha256)):
        raise ValueError("La ruta, el commit o el hash no son válidos.")
    destination = Path(destination)
    if destination.exists():
        if destination.is_symlink() or hashlib.sha256(destination.read_bytes()).hexdigest() != expected_sha256:
            raise RuntimeError("Archivo local distinto; se conserva: " + str(destination))
        return expected_sha256
    url = f"https://raw.githubusercontent.com/{REPOSITORY}/{commit}/{relative_path}"
    with urllib.request.urlopen(url, timeout=120) as response:
        payload = response.read(32 * 1024 * 1024 + 1)
    if len(payload) > 32 * 1024 * 1024 or hashlib.sha256(payload).hexdigest() != expected_sha256:
        raise RuntimeError("Tamaño o SHA-256 distinto para " + relative_path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".download")
    with temporary.open("xb") as stream:
        stream.write(payload)
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(temporary, destination)
    return expected_sha256


def load_module(name, path):
    path = Path(path).resolve()
    existing = sys.modules.get(name)
    if existing is not None:
        if Path(existing.__file__).resolve() != path:
            raise RuntimeError("Hay otra versión del helper cargada; reinicia la sesión antes de continuar.")
        return existing
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


In [ ]:
for relative_path, digest in SOURCE_FILE_SHA256.items():
    download_pinned(relative_path, SOURCE_COMMIT, SOURCE_DIR / relative_path, digest)
# Verificar también el helper transitivo si otra celda ya lo dejó cargado.
for alias in ("scripts.e3_github_storage", "e3_github_storage"):
    cached = sys.modules.get(alias)
    if cached is not None:
        cached_path = Path(cached.__file__)
        if hashlib.sha256(cached_path.read_bytes()).hexdigest() != SOURCE_FILE_SHA256["scripts/e3_github_storage.py"]:
            raise RuntimeError("Hay otra versión del almacenamiento cargada; reinicia la sesión antes de continuar.")
GITHUB_HELPER = load_module("e3_innovation_github_colab_v2", SOURCE_DIR / "scripts/e3_innovation_github.py")
if hashlib.sha256(Path(GITHUB_HELPER.core.__file__).read_bytes()).hexdigest() != SOURCE_FILE_SHA256["scripts/e3_github_storage.py"]:
    raise RuntimeError("El almacenamiento transitivo no coincide con el código verificado.")
print("Código descargado y verificado:", len(SOURCE_FILE_SHA256), "archivos")


## Autenticación

El coordinador recibe el secreto en memoria. El entorno del trabajador científico filtra credenciales. Una falta de acceso o una publicación sin confirmar detiene el avance.


In [ ]:
GITHUB_AUTH_TOKEN = os.environ.get("GITHUB_TOKEN", "").strip()
if not GITHUB_AUTH_TOKEN:
    try:
        from google.colab import userdata
        GITHUB_AUTH_TOKEN = (userdata.get("GITHUB_TOKEN") or "").strip()
    except Exception:
        GITHUB_AUTH_TOKEN = ""
if not GITHUB_AUTH_TOKEN:
    from getpass import getpass
    GITHUB_AUTH_TOKEN = getpass("GITHUB_TOKEN con Contents: Read and write para este repositorio: ").strip()
if not GITHUB_AUTH_TOKEN or any(character.isspace() for character in GITHUB_AUTH_TOKEN):
    raise RuntimeError("Configura un GITHUB_TOKEN válido y vuelve a ejecutar esta celda.")
print("Credencial recibida en memoria.")


## Preparar el entorno aislado, comprobar las matemáticas y descargar los datos

Se usa Python 3.12.12, PyTorch 2.8.0+cu126, NumPy 2.3.5 y SciPy 1.17.0. La comprobación sintética no abre datos de mercado ni ajusta parámetros. El NPZ se verifica por SHA-256; sólo se admiten sus tres arreglos originales de entrenamiento. Las particiones de calibración y evaluación se reservan.


In [ ]:
runtime_helper = load_module("e3_innovacion_runtime_v2", SOURCE_DIR / "notebooks/preparar_e3_innovacion_runtime.py")
GPU_PYTHON, RUN_ENV, GPU_RUNTIME = runtime_helper.preparar_entorno(
    LOCAL_WORKSPACE / "runtime", device=DEVICE)
subprocess.run([str(GPU_PYTHON), str(SOURCE_DIR / "scripts/e3_verify_innovation_design.py"),
                "--root", str(SOURCE_DIR), "--check"], env=RUN_ENV, check=True)
PLAN_PATH = SOURCE_DIR / "experiments/e3_innovacion_v2_20261116/plan.json"
PLAN = json.loads(PLAN_PATH.read_text())
data_source = PLAN["data"]
DATA_PATH = LOCAL_WORKSPACE / "datos_auditados" / "current_cohort_arrays.npz"
DATA_SHA256 = data_source["sha256"]
download_pinned(data_source["path"], data_source["commit"], DATA_PATH, DATA_SHA256)
print("Datos verificados:", DATA_SHA256)
print("Ajuste / purga / validación previstos:", data_source["fit_count"],
      data_source["purged_count"], data_source["validation_count"])


## Recuperar o inicializar la misma corrida

Esta celda confirma el registro remoto antes de cualquier ajuste de medias, escalas o biases. La inicialización **consume presupuesto** y conserva el cargo de v1. El M00 anterior se recupera de su snapshot inmutable y se verifica antes de usarlo como control.

Una reserva activa de otra sesión impide entrenar simultáneamente. Después de un cierre abrupto se conserva la fecha activa y se aplica el cargo conservador al recuperar. No abras varias sesiones para esta corrida.


In [ ]:
GITHUB_ARGS = dict(token=GITHUB_AUTH_TOKEN, npz_path=DATA_PATH, expected_sha256=DATA_SHA256,
                   run_dir=RUN_DIR, plan_path=PLAN_PATH, code_root=SOURCE_DIR,
                   device=DEVICE, python=GPU_PYTHON, environment=RUN_ENV)
preparation = GITHUB_HELPER.prepare_github_innovation(**GITHUB_ARGS)
print(json.dumps(preparation, ensure_ascii=False, indent=2))
print("Archivos y presupuesto conservados en:", RESULTS_URL)


## Entrenar, validar y guardar en GitHub

Se intercalan épocas del candidato, la ablación sin NLL auxiliar y la cabeza positiva aditiva. Las tres redes mantienen 1210 parámetros, una semilla, hasta 40 épocas y paciencia 8. Se comparan además con el piso solo, piso más incremento medio de ajuste y M00 v1 proyectado.

Cada bloque requiere confirmación remota ACTIVE antes de entrenar y CLOSED antes del siguiente. Se conservan pesos, optimizador, generadores aleatorios, curvas y predicciones. Ejecutar de nuevo recupera la corrida existente; no concede otro presupuesto. La publicación añade tiempo de pared y los cargos aplicados durante una reserva activa también consumen el saldo.


In [ ]:
execution = GITHUB_HELPER.run_github_innovation(**GITHUB_ARGS)
print(json.dumps(execution, ensure_ascii=False, indent=2))
print("Resultados y checkpoints:", RESULTS_URL)


## Revisar los resultados exploratorios

La tabla muestra RMSE y MAE en las unidades originales de σ₇. Los mejores checkpoints se seleccionan con la misma validación expuesta de 116 fechas. La cobertura de las cabezas probabilísticas es nominal bajo su familia asumida; no se presenta como calibrada.


In [ ]:
from html import escape
from IPython.display import HTML, display

summary_path = RUN_DIR / "summary.json"
if not summary_path.is_file():
    print("Aún no hay resumen. Revisa el estado remoto y el registro de la corrida.")
else:
    summary = json.loads(summary_path.read_text())
    print("Estado:", summary.get("status"))
    print(json.dumps({key: value for key, value in summary.items()
                      if "second" in key or "budget" in key or key == "decision"},
                     ensure_ascii=False, indent=2))
    rows = []
    for name, result in summary.get("models", {}).items():
        values = [name, result.get("parameters"), result.get("completed_epochs"),
                  result.get("best_epoch"), result.get("RMSE"),
                  result.get("MAE")]
        rows.append("<tr>" + "".join("<td>" + escape(str(value)) + "</td>" for value in values) + "</tr>")
    display(HTML("<table><thead><tr><th>Variante o control</th><th>Parámetros</th>"
                 "<th>Épocas completas</th><th>Mejor época</th><th>RMSE validación</th>"
                 "<th>MAE validación</th></tr></thead><tbody>" + "".join(rows) + "</tbody></table>"))
    print("El resumen completo y las métricas secundarias están en", RESULTS_URL)


In [ ]:
import csv
try:
    import plotly.graph_objects as go
except ImportError:
    print("Las curvas están disponibles en los CSV de GitHub.")
else:
    figure = go.Figure()
    for name in ("innovation_density_mean", "innovation_mean_only", "additive_positive_head"):
        curves_path = RUN_DIR / name / "curves.csv"
        if curves_path.is_file():
            with curves_path.open() as stream:
                rows = list(csv.DictReader(stream))
            figure.add_trace(go.Scatter(x=[int(row["epoch"]) for row in rows],
                                       y=[float(row["validation_RMSE_original"]) for row in rows],
                                       mode="lines+markers", name=name))
    if summary_path.is_file():
        summary = json.loads(summary_path.read_text())
        for name in ("floor_only", "floor_plus_fit_mean", "frozen_M00_projected"):
            value = summary.get("models", {}).get(name, {}).get("RMSE")
            if value is not None:
                figure.add_hline(y=value, line_dash="dot", annotation_text=name)
    figure.update_layout(title="Validación interna expuesta · v2 · una semilla",
                         xaxis_title="Época", yaxis_title="RMSE de σ₇ (unidades originales)",
                         template="plotly_white")
    figure.show()


## Decisión posterior

La heurística de continuación exige al candidato **al menos 2 % menos RMSE que el mejor de los cinco controles**, con las tres recetas neuronales completas. Ese umbral no es un margen confirmatorio ni una prueba de significancia. Un piloto incompleto no supera la heurística.

Una continuación requiere revisar la novedad, comprender y defender el código, congelar el protocolo, calcular potencia y preparar la comparación formal con el Top 10 y los baselines. Una mejora en este piloto no certifica R1/R3. La asistencia de IA debe declararse en el informe.
